# Анализ данных: исправленные пары «чертёж — STL»

## Кратко

**Вход** — один PNG полного листа с видами, размерами и пояснениями,
а также ручная подсказка ориентации детали из манифеста.
**Ожидаемый выход** — геометрия детали в STL, в миллиметрах.
Исследуем **10 деталей dev исследования**: что видно на чертежах, как различаются формы
и размеры, пригодны ли сетки для сравнения. Пять val описаны отдельно в [EXPERIMENTS.md](../EXPERIMENTS.md).

Источник — исправленный выпуск `20261115-v2` учебных примеров КОМПАС-3D.
Из 27 исходных пар выбраны однолистовые варианты, принятые для номинальной геометрии.
Часть размеров дописана по CAD-моделям; это подготовленный учебный набор, не независимое
восстановление исходных неразмеченных чертежей. Отбор не использовал результаты генератора.

Ниже только **входные данные и эталоны**. Результаты запусков агента и его метрики
в этот анализ не входят. Файлы читаются из [манифеста](../data/manifest.json);
роли сверяются с [манифестом экспериментов](../data/manifest-experiments.json). Набор не меняется,
запросов к API нет. Исходный ноутбук ДЗ1 сохранён в локальном архиве перед заменой данных.


In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import trimesh
from IPython.display import Markdown, display
from matplotlib.colors import LightSource
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from PIL import Image

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/manifest.json").is_file())
DATA = ROOT / "data"
manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
items = {item["pair_id"]: item for item in manifest["items"]}
experiments = json.loads((DATA / "manifest-experiments.json").read_text(encoding="utf-8"))
dev = {item["pair_id"]: item for item in experiments["items"] if item["split"] == "dev"}
assert len(items) == 10 and items.keys() == dev.keys()
assert all(item["sheet_count"] == 1 for item in items.values())
# В совместимом manifest.json роль val нужна прежнему оценщику. В экспериментах это dev.
for key, item in items.items():
    assert item["input_png"] == dev[key]["input_png"]
    assert item["target_mesh"] == dev[key]["target_mesh"]
for item in items.values():
    for key in ("input_png", "target_mesh"):
        file = item[key]
        assert hashlib.sha256((DATA / file["path"]).read_bytes()).hexdigest() == file["sha256"], file["path"]

plt.rcParams.update({"font.size": 11, "figure.dpi": 150, "figure.facecolor": "white"})
print(f"Доступны {len(items)} пар. Контрольные суммы PNG и STL совпадают с манифестом датасета.")

## 1. Вход: общий вид чертежей

Одна галерея **4–4–2** показывает состав и компоновку листов. Масштаб миниатюр выбран
для обзора; мелкие размеры следует читать в оригиналах по ссылкам под галереей.
Цвета, пропорции и весь лист сохранены, обрезки видов нет.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 11.5), dpi=220, layout="constrained")
for axis in axes.flat:
    axis.set_axis_off()
for axis, (key, item) in zip(axes.flat, items.items(), strict=False):
    with Image.open(DATA / item["input_png"]["path"]) as picture:
        preview = picture.copy()
        preview.thumbnail((1600, 1600), Image.Resampling.LANCZOS)
    axis.imshow(preview)
    axis.set_title(f"{key} · {item['name']}", fontsize=10, pad=8)
plt.show()
display(Markdown("Оригиналы: " + " · ".join(
    f"[{key}](../data/{item['input_png']['path']})" for key, item in items.items()
)))

## 2. Размер файла и объём изображения — разные вещи

Проверяем формат и полное декодирование каждого PNG. Считаем разрешение, размер файла
и долю небелых пикселей: яркость ниже 245 из 255 после перевода в оттенки серого.
В эту долю входят контуры, текст, штриховка и рамка; это **не площадь детали**
и не автоматическая оценка читаемости размеров.

In [ ]:
image_rows = []
for key, item in items.items():
    path = DATA / item["input_png"]["path"]
    with Image.open(path) as picture:
        picture.load()
        assert picture.format == "PNG" and picture.mode == "RGB"
        width, height = picture.size
        ink_percent = float((np.asarray(picture.convert("L")) < 245).mean() * 100)
    image_rows.append({
        "Деталь": key, "Разрешение, пикс.": f"{width} × {height}",
        "Мегапиксели": width * height / 1_000_000,
        "PNG, КиБ": path.stat().st_size / 1024,
        "Небелые пиксели, %": ink_percent,
        "Ориентация": "альбомная" if width > height else "книжная",
    })
images = pd.DataFrame(image_rows).set_index("Деталь")
display(images.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), layout="constrained")
for axis, column, color in zip(axes, ["Мегапиксели", "PNG, КиБ"], ["#527c98", "#709783"], strict=False):
    values = images[column].sort_values()
    axis.barh(values.index, values, color=color)
    axis.set_xlabel(column)
    axis.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
largest = images["Мегапиксели"].nlargest(2)
display(Markdown(
    f"**Наблюдения.** Все {len(images)} входов — читаемые RGB PNG. "
    f"Разрешение меняется от {images['Мегапиксели'].min():.1f} до "
    f"{images['Мегапиксели'].max():.1f} Мп, а файлы занимают всего "
    f"{images['PNG, КиБ'].min():.0f}–{images['PNG, КиБ'].max():.0f} КиБ. "
    f"Самые большие растры — {', '.join(largest.index)}. "
    f"Небелые пиксели занимают {images['Небелые пиксели, %'].min():.1f}–"
    f"{images['Небелые пиксели, %'].max():.1f}% листа.\n\n"
))

Однородный белый фон хорошо сжимается: небольшой PNG может требовать много памяти после декодирования. Размер в байтах сам по себе не характеризует нагрузку обработки. При уменьшении листа мелкие надписи и знаки допусков теряют читаемость; предобработку нужно учитывать в протоколе, сохраняя исходные PNG.

**Что нужно извлекать из этих изображений.** На одном листе совмещены разные виды,
разрезы, размерные выноски и служебные надписи. Размеры нельзя восстанавливать по
масштабу картинки: нужны числовые значения и связь между видами. Контуры, осевые линии
и надписи используют разные цвета; перевод в чёрно-белый вид может убрать полезные различия.

| Наблюдение в наборе | Почему это важно для восстановления |
|---|---|
| Зубчатые детали KF0011, KF0012, KF0025 и KF0029 | Венец задаётся в том числе таблицей параметров, а не только нарисованным контуром. |
| Колесо со спицами KF0029 и деталь с отверстиями KF0031 | Нужно согласовать ступени, окна, отверстия и осевые глубины. Одного внешнего силуэта недостаточно. |
| Втулки, сальник, шайба и кольцо | Простая общая форма всё равно содержит пазы, фаски, зенковку или отверстия. |
| Допуски и справочные размеры | Номинал, интервал допуска и размер со звёздочкой имеют разный смысл. |

В дополненной версии недостающие размеры перенесены на тот же лист. Примечания
являются частью входа: в них D означает диаметр. Это качественный обзор листов
и сохранённых проверок пар, не результат OCR.
Полнота всех размерных цепей не доказана: часть геометрии задаётся таблицами и условными
обозначениями. Масок видов, распознанного текста и полной разметки размеров в наборе нет.

## 3. Выход: эталонные STL

STL хранит поверхность из треугольников. В нём нет дерева построения, семантики отверстий
или единиц измерения — **миллиметры заданы манифестом датасета**. Анализируем именно подготовленные
сетки `data/target`, с которыми затем сравнивается результат генератора.

Общая система координат: основная цилиндрическая ось совпадает с Z и проходит через
(0, 0) в XY, базовый торец — на Z=0, весь материал при Z≥0. Для каждой детали поле
`orientation` задаёт торец, направление +Z и угловой ориентир +Y через видимые
элементы чертежа. X определяется правой системой с учётом направления взгляда.
Эту ручную подсказку одинаково получают все маршруты генератора.

У KF0012 снизу крайний торец короткого выступа ступицы, сверху длинный выступ,
паз направлен к +Y. Для этого STL повёрнут на 180° вокруг Y и перенесён по Z;
его форма и размеры сохранены с точностью записи STL. Остальные девять сеток dev
не изменены относительно источника. Матрица и хеш исходника KF0012 — в `preparation`
манифеста. Наличие общей оси не означает осевую симметрию: её нарушают пазы,
отверстия и спицы. Центровка по габаритному ящику эту ось не заменяет.


In [ ]:
assert manifest["coordinates"]["units"] == "mm"
meshes, mesh_rows = {}, []
for key, item in items.items():
    target = item["target_mesh"]
    mesh = trimesh.load_mesh(DATA / target["path"], process=True)
    assert isinstance(mesh, trimesh.Trimesh) and np.isfinite(mesh.vertices).all()
    meshes[key] = mesh
    x, y, z = mesh.extents
    mesh_rows.append({
        "Деталь": key, "X, мм": x, "Y, мм": y, "Z, мм": z,
        "Объём, см³": mesh.volume / 1000, "Треугольники": len(mesh.faces),
        "Замкнута": "да" if mesh.is_watertight else "нет",
        "Корректный объём": "да" if mesh.is_volume else "нет",
        "Компоненты": mesh.body_count,
        "Z минимум, мм": mesh.bounds[0, 2],
    })
models = pd.DataFrame(mesh_rows).set_index("Деталь")
display(models.drop(columns=["Z минимум, мм"]).round(2))
print(f"Максимальное отклонение базового торца от Z=0: {models['Z минимум, мм'].abs().max():.2g} мм.")

In [ ]:
fig = plt.figure(figsize=(16, 10), dpi=180, layout="constrained")
elevation, azimuth = np.deg2rad([35, -65])
direction = [np.cos(elevation) * np.cos(azimuth), np.cos(elevation) * np.sin(azimuth), np.sin(elevation)]
for index, (key, mesh) in enumerate(meshes.items(), start=1):
    axis = fig.add_subplot(3, 4, index, projection="3d")
    triangles = mesh.triangles
    # Убираем обратные стороны только при отрисовке; исходная сетка не меняется.
    visible = mesh.face_normals @ direction > 0
    axis.add_collection3d(Poly3DCollection(
        triangles[visible], facecolors="#9bb5c8", linewidths=0,
        antialiased=False, shade=True, lightsource=LightSource(315, 45),
    ))
    center = mesh.bounds.mean(axis=0)
    radius = mesh.extents.max() / 2
    axis.set(xlim=(center[0] - radius, center[0] + radius),
             ylim=(center[1] - radius, center[1] + radius),
             zlim=(center[2] - radius, center[2] + radius), title=f'{key} \n{items[key]["name"]}')
    axis.set_box_aspect((1, 1, 1))
    axis.set_proj_type("ortho")
    axis.view_init(elev=35, azim=-65)
    axis.set_axis_off()
fig.suptitle("Эталоны STL · масштаб каждой миниатюры подобран отдельно", fontsize=14)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), layout="constrained")
for axis, column, color in zip(axes, ["Объём, см³", "Треугольники"], ["#709783", "#527c98"], strict=False):
    values = models[column].sort_values()
    axis.barh(values.index, values, color=color)
    axis.set_xscale("log")
    axis.set_xlabel(column + " · логарифмическая шкала")
    axis.spines[["top", "right"]].set_visible(False)
plt.show()

display(Markdown(
    f"Все {len(models)} сеток замкнуты: **{(models['Замкнута'] == 'да').sum()}/{len(models)}**; "
    f"корректный объём имеют **{(models['Корректный объём'] == 'да').sum()}/{len(models)}**. "
    f"Число связных компонентов: {models['Компоненты'].min()}–{models['Компоненты'].max()}. "
    f"Объёмы отличаются в **{models['Объём, см³'].max() / models['Объём, см³'].min():.0f} раз**, "
    f"число треугольников — от **{models['Треугольники'].min():,}** "
    f"до **{models['Треугольники'].max():,}**.\n\n"
))

Сетки пригодны для геометрического сравнения. Замкнутость подтверждает техническую целостность поверхности, но не соответствие чертежу. Число треугольников зависит от кривизны и настроек выгрузки: это не число конструктивных элементов и не оценка сложности задания. При оценке детали должны иметь одинаковый вес, иначе крупные или густо разбитые на треугольники модели будут определять итог.

## 4. Что исправлено и какие ограничения остались

Контрольная сумма подтверждает сохранность файла, но не правильность пары.
Перед переносом выбраны пары, принятые в источнике для номинальной геометрии;
проверены количество листов, файлы, координаты и целостность сеток.
Это не сплошная экспертиза каждого допуска и всех поверхностей.
Исходные модели и чертежи сохранены отдельно, изменения выполнены до запусков исследования.


In [ ]:
counts = pd.Series([item["pair_validation"] for item in items.values()]).value_counts()
display(counts.rename_axis("Статус проверки").to_frame("Пар"))

| Пара | Было в исходной версии | Стало в текущем эталоне / на листе |
|---|---|---|
| KF0011 | Отверстие Ø30,6 при Ø32 на листе | Отверстие Ø32; исправлено положение дна паза |
| KF0012 | Отверстие Ø55 при Ø54; положение дна паза 59,5 | Отверстие Ø54; положение дна паза 58,3 |
| KF0018 | Конические торцы, осевой габарит около 7,10644 мм | Плоское кольцо Ø81/Ø64 × 6 мм |
| KF0025, KF0029, KF0031 | Не все основные размеры были на входном листе | Размеры дополнены на том же листе по CAD-модели |

У KF0012 остаётся наружный габарит около 245,864 мм при Ø245h12 на листе;
исправление отверстия не устранило это различие. Эта пара находится только в dev.
У KF0018 прежние Ø65 и Ø82 лежали на границах допусков Ø64±1 и Ø81±1 — это не выход
за допуск. Толщина 6* справочная; для учебного сравнения выбран плоский номинальный вариант.

Зубья, фаски, резьбы и мелкие скругления сохранены: текущий рецепт не всегда способен
точно выразить их. Положение зубьев относительно пазов/крепежа местами не размерено;
подбирать его по метрикам нельзя. Для KF0027 фиксировано состояние после сборочного
сверления; отверстия нельзя заменять равномерным круговым массивом.
Подробности по каждой паре — в полях `changes`, `orientation` и `limitations`
[манифеста](../data/manifest.json).

Следствие для эксперимента: готовый файл с корректным объёмом ещё не гарантирует
идеальную достижимость IoU и F-score. Различаем оставшиеся ограничения исходной пары,
ограничения рецепта и ошибки чтения чертежа. Старые оценки ДЗ1 не относятся к новым STL.


## 5. Выводы и ограничения эксперимента

1. **Вход неоднороден по разрешению и содержанию.** От простых колец переходим к
   зубчатым колёсам, спицам, ступеням и группам отверстий. Нужны числовые размеры,
   виды, разрезы и таблицы. Уменьшение листа может ухудшать читаемость мелких примечаний.
2. **Выход позволяет обойтись без CAD-приложения.** STL можно читать, показывать
   и проверять Python-библиотеками. Единицы и базовый торец фиксируются отдельно;
   центровка по габаритному ящику не заменяет основную ось детали.
3. **Простота рецепта ограничивает достижимую точность.** Фаски, скругления, зенковки
   и зубчатые поверхности лишь приближаются выдавливаниями. Это не обязательно
   ошибка чтения чертежа; сведения о способе моделирования резьб сохраняются.
4. **Крупные известные расхождения исправлены, ограничения остаются явными.**
   Для KF0012 сохраняется замечание к наружному диаметру. Дописанные по модели размеры
   делают вход полнее, но ограничивают перенос выводов на исходные неполные чертежи.
5. **Это десять деталей для отладки, не независимый тест.** Пять новых val заданы
   отдельно. Близкие детали не распределяются между dev и val, папка источника
   сама по себе не определяет родство. Выводов о всём корпусе и качестве генератора
   по этому EDA не делаем.

Этот EDA устанавливает свойства входов, пригодность эталонных сеток и ограничения данных.
Качество генератора проверяется отдельным экспериментом.
